In [ ]:

!pip -q install -U datasets

from datasets import load_dataset

dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

... (3 outputs left out)

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [ ]:

print("Available splits:", list(dataset.keys()))

for split in dataset:
    print(split, ":", len(dataset[split]), "sentence pairs")

print("\nFirst English-Hindi example:")

example = dataset["train"][0]["translation"]

print("English:", example["en"])
print("Hindi:", example["hi"])


Available splits: ['train', 'validation', 'test']
train : 1659083 sentence pairs
validation : 520 sentence pairs
test : 2507 sentence pairs

First English-Hindi example:
English: Give your application an accessibility workout
Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें


In [ ]:

import pandas as pd
import numpy as np
import pyarrow
import datasets
import transformers
import torch

print("All core imports successful!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyArrow:", pyarrow.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())


All core imports successful!
Pandas: 2.2.3
NumPy: 2.2.6
PyArrow: 26.0.0
Datasets: 5.1.0
Transformers: 5.22.0
GPU available: True


In [ ]:
from datasets import load_dataset

dataset = load_dataset("cfilt/iitb-english-hindi")

print("Available splits:", list(dataset.keys()))

for split in dataset.keys():
    print(f"{split}: {len(dataset[split]):,} sentence pairs")

print("\nSample English-Hindi pair:")
print("English:", dataset["train"][0]["translation"]["en"])
print("Hindi:  ", dataset["train"][0]["translation"]["hi"])

Available splits: ['train', 'validation', 'test']
train: 1,659,083 sentence pairs
validation: 520 sentence pairs
test: 2,507 sentence pairs

Sample English-Hindi pair:
English: Give your application an accessibility workout
Hindi:   अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें


In [ ]:
import re
import unicodedata
import random

# Reproducibility
SEED = 42
random.seed(SEED)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Clean the training data
train_data = []
seen_pairs = set()

for item in dataset["train"]:
    en = clean_text(item["translation"]["en"])
    hi = clean_text(item["translation"]["hi"])

    # Remove empty pairs
    if not en or not hi:
        continue

    # Remove unusually long sentences
    if len(en.split()) > 100 or len(hi.split()) > 100:
        continue

    # Remove exact duplicate pairs
    pair = (en, hi)
    if pair in seen_pairs:
        continue

    seen_pairs.add(pair)
    train_data.append({"en": en, "hi": hi})

print("Clean training pairs:", f"{len(train_data):,}")

# Shuffle reproducibly and select a manageable subset
random.Random(SEED).shuffle(train_data)
train_sample = train_data[:40_000]

print("Selected training pairs:", f"{len(train_sample):,}")
print("\nExample after cleaning:")
print("English:", train_sample[0]["en"])
print("Hindi:  ", train_sample[0]["hi"])

Clean training pairs: 1,404,813
Selected training pairs: 40,000

Example after cleaning:
English: Jaipur is the State capital.
Hindi:   जयपुर राज्य की राजधानी है।


In [ ]:
from datasets import Dataset

# Reserve 40,000 pairs for training and 2,000 separate pairs for validation
train_pairs = train_sample[:40_000]
validation_pairs = train_data[40_000:42_000]

# Keep the official test set separate
test_pairs = []

for item in dataset["test"]:
    en = clean_text(item["translation"]["en"])
    hi = clean_text(item["translation"]["hi"])

    if en and hi and len(en.split()) <= 100 and len(hi.split()) <= 100:
        test_pairs.append({"en": en, "hi": hi})

# Convert to Hugging Face datasets
train_ds = Dataset.from_list(train_pairs)
validation_ds = Dataset.from_list(validation_pairs)
test_ds = Dataset.from_list(test_pairs)

print("Training pairs:", len(train_ds))
print("Validation pairs:", len(validation_ds))
print("Test pairs:", len(test_ds))

print("\nTraining example:")
print(train_ds[0])

Training pairs: 40000
Validation pairs: 2000
Test pairs: 2507

Training example:
{'en': 'Jaipur is the State capital.', 'hi': 'जयपुर राज्य की राजधानी है।'}


In [ ]:
from collections import Counter
import re

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

def tokenize_en(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

def tokenize_hi(text):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

# Build vocabularies from training data only
en_counter = Counter()
hi_counter = Counter()

for row in train_ds:
    en_counter.update(tokenize_en(row["en"]))
    hi_counter.update(tokenize_hi(row["hi"]))

MIN_FREQ = 2

en_vocab = special_tokens + [
    word for word, count in en_counter.items() if count >= MIN_FREQ
]
hi_vocab = special_tokens + [
    word for word, count in hi_counter.items() if count >= MIN_FREQ
]

en_stoi = {token: idx for idx, token in enumerate(en_vocab)}
hi_stoi = {token: idx for idx, token in enumerate(hi_vocab)}

print("English vocabulary size:", len(en_vocab))
print("Hindi vocabulary size:", len(hi_vocab))
print("Special token IDs:", {
    "PAD": en_stoi[PAD_TOKEN],
    "UNK": en_stoi[UNK_TOKEN],
    "BOS": en_stoi[BOS_TOKEN],
    "EOS": en_stoi[EOS_TOKEN]
})

English vocabulary size: 18015
Hindi vocabulary size: 4258
Special token IDs: {'PAD': 0, 'UNK': 1, 'BOS': 2, 'EOS': 3}


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset as TorchDataset, DataLoader

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

MAX_LEN = 40

def encode_sentence(text, vocab, tokenizer):
    tokens = tokenizer(text)
    ids = [vocab.get(token, UNK_ID) for token in tokens[:MAX_LEN - 2]]
    return [BOS_ID] + ids + [EOS_ID]

class TranslationDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = []
        for row in hf_dataset:
            src = encode_sentence(row["en"], en_stoi, tokenize_en)
            tgt = encode_sentence(row["hi"], hi_stoi, tokenize_hi)
            self.data.append((
                torch.tensor(src, dtype=torch.long),
                torch.tensor(tgt, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(
        src_batch, batch_first=True, padding_value=PAD_ID
    )
    tgt_batch = pad_sequence(
        tgt_batch, batch_first=True, padding_value=PAD_ID
    )
    return src_batch, tgt_batch

train_torch_ds = TranslationDataset(train_ds)
val_torch_ds = TranslationDataset(validation_ds)
test_torch_ds = TranslationDataset(test_ds)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_torch_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_torch_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_torch_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn
)

src_batch, tgt_batch = next(iter(train_loader))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Source batch shape:", src_batch.shape)
print("Target batch shape:", tgt_batch.shape)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

Training batches: 625
Validation batches: 32
Test batches: 40
Source batch shape: torch.Size([64, 40])
Target batch shape: torch.Size([64, 40])
Device: CUDA


In [ ]:

import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        positions = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerTranslator(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        nhead=4,
        num_layers=3,
        dim_feedforward=512,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(
            src_vocab_size, d_model, padding_idx=PAD_ID
        )
        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size, d_model, padding_idx=PAD_ID
        )

        self.positional_encoding = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.output_layer = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt_input):
        src_padding_mask = src.eq(PAD_ID)
        tgt_padding_mask = tgt_input.eq(PAD_ID)

        tgt_len = tgt_input.size(1)
        causal_mask = torch.triu(
            torch.ones(tgt_len, tgt_len, device=tgt_input.device,
                       dtype=torch.bool),
            diagonal=1
        )

        src_emb = self.positional_encoding(
            self.src_embedding(src) * math.sqrt(self.d_model)
        )
        tgt_emb = self.positional_encoding(
            self.tgt_embedding(tgt_input) * math.sqrt(self.d_model)
        )

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=causal_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        return self.output_layer(output)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerTranslator(
    src_vocab_size=len(en_vocab),
    tgt_vocab_size=len(hi_vocab)
).to(device)

parameter_count = sum(p.numel() for p in model.parameters())

print(model)
print(f"\nTotal parameters: {parameter_count:,}")
print(f"Model device: {next(model.parameters()).device}")


TransformerTranslator(
  (src_embedding): Embedding(18015, 256, padding_idx=0)
  (tgt_embedding): Embedding(4258, 256, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affi

In [ ]:

import time
import copy
import torch
import torch.nn as nn

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.0003
)

EPOCHS = 5
best_val_loss = float("inf")
history = []
best_model_state = None

def run_epoch(loader, training=True):
    model.train(training)
    total_loss = 0.0
    total_tokens = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        # Input excludes final token; labels exclude initial token
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(src, tgt_input)

            loss_sum = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

            # CrossEntropyLoss above returns mean over non-padding tokens.
            # Weight batch loss by its number of target tokens.
            token_count = tgt_output.ne(PAD_ID).sum().item()
            loss = loss_sum

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )
                optimizer.step()

        total_loss += loss.item() * token_count
        total_tokens += token_count

    return total_loss / max(total_tokens, 1)


start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = run_epoch(train_loader, training=True)

    with torch.no_grad():
        val_loss = run_epoch(val_loader, training=False)

    epoch_seconds = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "epoch_seconds": epoch_seconds
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {epoch_seconds / 60:.2f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, "transformer_baseline_best.pt")
        print("  Best checkpoint saved.")

total_training_seconds = time.time() - start_time

# Restore the best validation checkpoint
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print(f"\nTotal training time: {total_training_seconds / 60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/5 | Train Loss: 3.6204 | Val Loss: 3.1838 | Time: 0.45 min
  Best checkpoint saved.
Epoch 2/5 | Train Loss: 3.1122 | Val Loss: 2.9360 | Time: 0.43 min
  Best checkpoint saved.
Epoch 3/5 | Train Loss: 2.9248 | Val Loss: 2.7909 | Time: 0.45 min
  Best checkpoint saved.
Epoch 4/5 | Train Loss: 2.7952 | Val Loss: 2.6797 | Time: 0.46 min
  Best checkpoint saved.
Epoch 5/5 | Train Loss: 2.6940 | Val Loss: 2.6006 | Time: 0.47 min
  Best checkpoint saved.

Total training time: 2.28 minutes
Best validation loss: 2.6006


In [ ]:

import torch

def translate_transformer(text, max_new_tokens=40):
    model.eval()

    src_ids = encode_sentence(text, en_stoi, tokenize_en)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    generated = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(src, generated)
            next_token = logits[:, -1, :].argmax(dim=-1).item()

            if next_token == EOS_ID:
                break

            generated = torch.cat([
                generated,
                torch.tensor([[next_token]], device=device)
            ], dim=1)

            if generated.size(1) >= MAX_LEN:
                break

    # Convert generated IDs back to Hindi tokens
    tokens = []
    for token_id in generated[0, 1:].tolist():
        if token_id == EOS_ID:
            break
        tokens.append(hi_vocab[token_id])

    return " ".join(tokens)


# Try a few examples
for i in range(5):
    source = test_ds[i]["en"]
    reference = test_ds[i]["hi"]
    prediction = translate_transformer(source)

    print(f"\nExample {i + 1}")
    print("English:   ", source)
    print("Reference: ", reference)
    print("Predicted: ", prediction)



Example 1
English:    A black box in your car?
Reference:  आपकी कार में ब्लैक बॉक्स?
Predicted:  क ् य ा आप क ् य ा आप क ् य ा आप क ् य ा आप क ् य ा आप क ् य ा ?

Example 2
English:    As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference:  जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Predicted:  क ि स ी क ा र ् य क े ल ि ए अपन े प ् रत ि शत ा र ् द े श क े ल ि ए क े ल ि ए

Example 3
English:    The devices, which track every mile a motorist drives and transmit that information to bureaucrats, are at the center of a controversial attempt in Washington and state planning offices to overhaul the outdated system for funding America's major roads.
Reference:  यह डिवा

In [ ]:

import re
import unicodedata

def tokenize_hi_fixed(text):
    text = unicodedata.normalize("NFC", text)
    # Preserve complete Unicode word sequences, including Devanagari marks
    return re.findall(r"[^\W_]+(?:['’-][^\W_]+)*|[^\w\s]", text, flags=re.UNICODE)

sample = "आपकी कार में ब्लैक बॉक्स?"

print("Original tokenizer:")
print(tokenize_hi(sample))

print("\nFixed tokenizer:")
print(tokenize_hi_fixed(sample))


Original tokenizer:
['आपक', 'ी', 'क', 'ा', 'र', 'म', 'े', 'ं', 'ब', '्', 'ल', 'ै', 'क', 'ब', 'ॉ', 'क', '्', 'स', '?']

Fixed tokenizer:
['आपक', 'ी', 'क', 'ा', 'र', 'म', 'े', 'ं', 'ब', '्', 'ल', 'ै', 'क', 'ब', 'ॉ', 'क', '्', 'स', '?']


In [ ]:

def tokenize_hi(text):
    # Preserve complete Hindi words and normalize whitespace.
    # Keep punctuation attached to its word for this baseline.
    text = unicodedata.normalize("NFC", text)
    return text.split()

sample = "आपकी कार में ब्लैक बॉक्स?"

print("Hindi tokens:", tokenize_hi(sample))
print("Reconstructed:", " ".join(tokenize_hi(sample)))


Hindi tokens: ['आपकी', 'कार', 'में', 'ब्लैक', 'बॉक्स?']
Reconstructed: आपकी कार में ब्लैक बॉक्स?


In [ ]:

from collections import Counter

hi_counter = Counter()

for row in train_ds:
    hi_counter.update(tokenize_hi(row["hi"]))

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]

hi_vocab = special_tokens + [
    word for word, count in hi_counter.items() if count >= 2
]

hi_stoi = {token: idx for idx, token in enumerate(hi_vocab)}

print("Corrected Hindi vocabulary size:", len(hi_vocab))
print("Sample Hindi tokens:", tokenize_hi(sample))


Corrected Hindi vocabulary size: 22753
Sample Hindi tokens: ['आपकी', 'कार', 'में', 'ब्लैक', 'बॉक्स?']


In [ ]:

# Rebuild encoded datasets using the corrected Hindi tokenizer

train_torch_ds = TranslationDataset(train_ds)
val_torch_ds = TranslationDataset(validation_ds)
test_torch_ds = TranslationDataset(test_ds)

train_loader = DataLoader(
    train_torch_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_torch_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_torch_ds,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

src_batch, tgt_batch = next(iter(train_loader))

print("Training pairs:", len(train_torch_ds))
print("Validation pairs:", len(val_torch_ds))
print("Test pairs:", len(test_torch_ds))
print("Corrected Hindi vocabulary:", len(hi_vocab))
print("Target batch shape:", tgt_batch.shape)
print("Maximum target token ID:", tgt_batch.max().item())
print("Valid target vocabulary range:", 0, len(hi_vocab) - 1)


Training pairs: 40000
Validation pairs: 2000
Test pairs: 2507
Corrected Hindi vocabulary: 22753
Target batch shape: torch.Size([64, 40])
Maximum target token ID: 22457
Valid target vocabulary range: 0 22752


In [ ]:

import torch
import torch.nn as nn
import math

# Reinitialize the Transformer with the corrected vocabulary
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

model = TransformerTranslator(
    src_vocab_size=len(en_vocab),
    tgt_vocab_size=len(hi_vocab),
    d_model=256,
    nhead=4,
    num_layers=3,
    dim_feedforward=512,
    dropout=0.1
).to(device)

parameter_count = sum(p.numel() for p in model.parameters())

print(f"English vocabulary: {len(en_vocab):,}")
print(f"Hindi vocabulary: {len(hi_vocab):,}")
print(f"Model parameters: {parameter_count:,}")
print(f"Device: {next(model.parameters()).device}")


English vocabulary: 18,015
Hindi vocabulary: 22,753
Model parameters: 20,238,817
Device: cuda:0


In [ ]:

import time
import copy

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

EPOCHS = 5
best_val_loss = float("inf")
history = []
best_model_state = None

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = run_epoch(train_loader, training=True)

    with torch.no_grad():
        val_loss = run_epoch(val_loader, training=False)

    epoch_seconds = time.time() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "epoch_seconds": epoch_seconds
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {epoch_seconds / 60:.2f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, "transformer_corrected_best.pt")
        print("  Best checkpoint saved.")

total_training_seconds = time.time() - start_time

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print(f"\nTotal training time: {total_training_seconds / 60:.2f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")


Epoch 1/5 | Train Loss: 6.5811 | Val Loss: 5.7718 | Time: 0.78 min
  Best checkpoint saved.
Epoch 2/5 | Train Loss: 5.8246 | Val Loss: 5.3830 | Time: 0.76 min
  Best checkpoint saved.
Epoch 3/5 | Train Loss: 5.4386 | Val Loss: 5.1466 | Time: 0.77 min
  Best checkpoint saved.
Epoch 4/5 | Train Loss: 5.1289 | Val Loss: 4.9776 | Time: 0.76 min
  Best checkpoint saved.
Epoch 5/5 | Train Loss: 4.8591 | Val Loss: 4.8606 | Time: 0.77 min
  Best checkpoint saved.

Total training time: 3.85 minutes
Best validation loss: 4.8606


In [ ]:

# Make sure the decoding function uses the corrected Hindi tokenizer/vocabulary.
# Greedy decoding predicts one token at a time.

for i in range(5):
    source = test_ds[i]["en"]
    reference = test_ds[i]["hi"]
    prediction = translate_transformer(source)

    print(f"\nExample {i + 1}")
    print("English:   ", source)
    print("Reference: ", reference)
    print("Predicted: ", prediction)




Example 1
English:    A black box in your car?
Reference:  आपकी कार में ब्लैक बॉक्स?
Predicted:  क्या आप अपने <unk> में <unk>

Example 2
English:    As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference:  जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Predicted:  <unk> के <unk> के <unk> के <unk> में <unk> के <unk> को <unk> के रूप में <unk> के रूप में <unk> के रूप में <unk> के रूप में <unk> के रूप में <unk> के <unk> के रूप में एक

Example 3
English:    The devices, which track every mile a motorist drives and transmit that information to bureaucrats, are at the center of a controversial attempt in Washington and state planning offices to overhaul the outdated system

In [ ]:

%pip -q install sacrebleu

import sacrebleu
import time

# Evaluate on the official test set
references = []
predictions = []

start_time = time.time()

for row in test_ds:
    prediction = translate_transformer(row["en"])
    predictions.append(prediction)
    references.append(row["hi"])

inference_seconds = time.time() - start_time

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print(f"Test examples: {len(references)}")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF: {chrf.score:.2f}")
print(f"Total inference time: {inference_seconds:.2f} seconds")
print(
    f"Average inference time: "
    f"{inference_seconds / len(references):.4f} seconds per sentence"
)


Test examples: 2507
BLEU: 0.15
chrF: 4.72
Total inference time: 462.96 seconds
Average inference time: 0.1847 seconds per sentence


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"

mt5_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mt5_model = mt5_model.to(device)

print("Model:", MODEL_NAME)
print("Device:", device)
print(
    "Parameters:",
    f"{sum(p.numel() for p in mt5_model.parameters()):,}"
)
print("Tokenizer loaded successfully.")


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Model: google/mt5-small
Device: cuda
Parameters: 300,176,768
Tokenizer loaded successfully.


In [ ]:
from datasets import Dataset
import torch

SOURCE_MAX_LEN = 128
TARGET_MAX_LEN = 128

def preprocess_mt5(batch):
    inputs = [
        "translate English to Hindi: " + text
        for text in batch["en"]
    ]

    model_inputs = mt5_tokenizer(
        inputs,
        max_length=SOURCE_MAX_LEN,
        truncation=True
    )

    labels = mt5_tokenizer(
        text_target=batch["hi"],
        max_length=TARGET_MAX_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

mt5_train = train_ds.map(
    preprocess_mt5,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing training data"
)

mt5_val = validation_ds.map(
    preprocess_mt5,
    batched=True,
    remove_columns=validation_ds.column_names,
    desc="Tokenizing validation data"
)

print("Training examples:", len(mt5_train))
print("Validation examples:", len(mt5_val))
print("Input keys:", mt5_train.column_names)
print("First tokenized example:", mt5_train[0].keys())

Tokenizing training data:   0%|          | 0/40000 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training examples: 40000
Validation examples: 2000
Input keys: ['input_ids', 'attention_mask', 'labels']
First tokenized example: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
import time
import torch
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=mt5_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_english_hindi",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
    seed=42
)

print("Training configuration ready.")
print("Effective batch size:", 8 * 2)
print("Epochs:", 2)
print("Training examples:", len(mt5_train))

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'

In [ ]:
import inspect
from transformers import Seq2SeqTrainingArguments

params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters

print("Evaluation argument available:",
      [x for x in params if "eval" in x and "strategy" in x])

print("Warmup arguments available:",
      [x for x in params if "warmup" in x])

print("Mixed precision arguments available:",
      [x for x in params if x in ("fp16", "bf16")])

print("Save arguments available:",
      [x for x in params if "save_strategy" in x or "save_steps" in x])

Evaluation argument available: ['eval_strategy']
Warmup arguments available: ['warmup_steps']
Mixed precision arguments available: ['bf16', 'fp16']
Save arguments available: ['save_strategy', 'save_steps']


In [ ]:
import time
import torch
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=mt5_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_english_hindi",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=250,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
    seed=42
)

print("Training configuration ready.")
print("Effective batch size:", 8 * 2)
print("Epochs:", 2)
print("Training examples:", len(mt5_train))

Training configuration ready.
Effective batch size: 16
Epochs: 2
Training examples: 40000


In [ ]:
trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=training_args,
    train_dataset=mt5_train,
    eval_dataset=mt5_val,
    data_collator=data_collator,
    processing_class=mt5_tokenizer
)

print("mT5 trainer initialized successfully.")

mT5 trainer initialized successfully.


In [ ]:
mt5_start_time = time.time()

mt5_train_result = trainer.train()

mt5_training_time = time.time() - mt5_start_time

trainer.save_model("./mt5_english_hindi/best_model")
mt5_tokenizer.save_pretrained("./mt5_english_hindi/best_model")

print(f"Total training time: {mt5_training_time / 60:.2f} minutes")
print("Model and tokenizer saved.")

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Total training time: 28.06 minutes
Model and tokenizer saved.


In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Model dtype:", next(mt5_model.parameters()).dtype)
print("Training loss:", mt5_train_result.training_loss)

print("\nChecking model parameters...")
bad_params = [
    name for name, param in mt5_model.named_parameters()
    if not torch.isfinite(param).all()
]
print("Parameters with NaN/Inf:", bad_params[:10])
print("Number of affected parameter tensors:", len(bad_params))

print("\nChecking training labels...")
for i in range(3):
    labels = mt5_train[i]["labels"]
    print(
        f"Example {i}:",
        "tokens =", len(labels),
        "valid labels =", sum(token != mt5_tokenizer.pad_token_id for token in labels)
    )

GPU: Tesla T4
Model dtype: torch.float32
Training loss: 0.0

Checking model parameters...
Parameters with NaN/Inf: []
Number of affected parameter tensors: 0

Checking training labels...
Example 0: tokens = 11 valid labels = 11
Example 1: tokens = 71 valid labels = 71
Example 2: tokens = 28 valid labels = 28


In [ ]:
import torch

mt5_model.eval()

batch = data_collator([mt5_train[i] for i in range(4)])
batch = {
    key: value.to(device) if isinstance(value, torch.Tensor) else value
    for key, value in batch.items()
}

print("Batch shapes:")
for key, value in batch.items():
    print(key, tuple(value.shape) if isinstance(value, torch.Tensor) else type(value))

with torch.no_grad():
    outputs = mt5_model(**batch)

print("\nDirect model loss:", outputs.loss.item())
print("Loss is finite:", torch.isfinite(outputs.loss).item())
print("Trainer training loss:", mt5_train_result.training_loss)
print("\nLast trainer log entries:")
print(trainer.state.log_history[-6:])

Batch shapes:
input_ids (4, 32)
attention_mask (4, 32)
labels (4, 72)
decoder_input_ids (4, 72)

Direct model loss: nan
Loss is finite: False
Trainer training loss: 0.0

Last trainer log entries:
[{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 3.168421052631579e-06, 'epoch': 1.88, 'step': 4700}, {'loss': 0.0, 'grad_norm': nan, 'learning_rate': 2.1157894736842107e-06, 'epoch': 1.92, 'step': 4800}, {'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1.0631578947368422e-06, 'epoch': 1.96, 'step': 4900}, {'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1.0526315789473686e-08, 'epoch': 2.0, 'step': 5000}, {'eval_loss': nan, 'eval_runtime': 15.0906, 'eval_samples_per_second': 132.533, 'eval_steps_per_second': 16.567, 'epoch': 2.0, 'step': 5000}, {'train_runtime': 1682.496, 'train_samples_per_second': 47.548, 'train_steps_per_second': 2.972, 'total_flos': 5377384356249600.0, 'train_loss': 0.0, 'epoch': 2.0, 'step': 5000}]


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

# Restore the original pretrained weights
del mt5_model
torch.cuda.empty_cache()

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/mt5-small",
    torch_dtype=torch.float32
).to(device)

mt5_model.config.use_cache = False
mt5_model.train()

# Test the loss on a small batch
batch = data_collator([mt5_train[i] for i in range(4)])
batch = {
    key: value.to(device)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

with torch.no_grad():
    outputs = mt5_model(**batch)

print("Fresh model loss:", outputs.loss.item())
print("Loss is finite:", torch.isfinite(outputs.loss).item())
print("Model dtype:", next(mt5_model.parameters()).dtype)
print("GPU memory allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fresh model loss: 32.44336700439453
Loss is finite: True
Model dtype: torch.float32
GPU memory allocated: 5.15 GB


In [ ]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import torch

data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=mt5_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_english_hindi_retry",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=False,
    fp16=False,
    bf16=False,
    report_to="none",
    dataloader_num_workers=2,
    seed=42
)

trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=training_args,
    train_dataset=mt5_train,
    eval_dataset=mt5_val,
    data_collator=data_collator,
    processing_class=mt5_tokenizer
)

print("Stable trainer configured.")
print("Batch size:", 4)
print("Gradient accumulation:", 4)
print("Effective batch size:", 16)
print("Learning rate:", 1e-5)
print("Mixed precision: disabled")

Stable trainer configured.
Batch size: 4
Gradient accumulation: 4
Effective batch size: 16
Learning rate: 1e-05
Mixed precision: disabled


In [ ]:
trainer.args.max_steps = 20
trainer.args.save_strategy = "no"
trainer.args.eval_strategy = "no"
trainer.args.logging_steps = 5

test_result = trainer.train()

print("Stability test loss:", test_result.training_loss)
print("Loss is finite:", torch.isfinite(
    torch.tensor(test_result.training_loss)
).item())

Step,Training Loss
5,120.810144
10,120.792993
15,127.030786
20,111.281299


Stability test loss: 119.97880554199219
Loss is finite: True


In [ ]:
import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM

# Clear the model changed by the stability test
del mt5_model
torch.cuda.empty_cache()

MODEL_NAME = "google/mt5-small"

mt5_config = AutoConfig.from_pretrained(MODEL_NAME)
mt5_config.tie_word_embeddings = False

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    config=mt5_config,
    dtype=torch.float32
).to(device)

mt5_model.config.use_cache = False
mt5_model.eval()

# Rebuild a fresh batch
batch = data_collator([mt5_train[i] for i in range(4)])
batch = {
    key: value.to(device)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

with torch.no_grad():
    outputs = mt5_model(**batch)

print("Initial loss:", outputs.loss.item())
print("Loss is finite:", torch.isfinite(outputs.loss).item())
print("Tied word embeddings:", mt5_model.config.tie_word_embeddings)
print("Model parameters:", f"{sum(p.numel() for p in mt5_model.parameters()):,}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

Initial loss: 14.699311256408691
Loss is finite: True
Tied word embeddings: False
Model parameters: 556,291,456


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

del mt5_model
torch.cuda.empty_cache()

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/mt5-small",
    dtype=torch.float32
).to(device)

mt5_model.config.use_cache = False

print("Parameters:", f"{sum(p.numel() for p in mt5_model.parameters()):,}")
print("Tied embeddings:", mt5_model.config.tie_word_embeddings)

batch = data_collator([mt5_train[i] for i in range(2)])
batch = {
    key: value.to(device)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

mt5_model.eval()
with torch.no_grad():
    outputs = mt5_model(**batch)

print("Initial loss:", outputs.loss.item())
print("Finite loss:", torch.isfinite(outputs.loss).item())

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Parameters: 300,176,768
Tied embeddings: True
Initial loss: 14.685127258300781
Finite loss: True


In [ ]:
import torch
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=mt5_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

test_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_stability_test",
    max_steps=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=0,
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=1,
    report_to="none",
    seed=42
)

test_trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=test_args,
    train_dataset=mt5_train,
    data_collator=data_collator,
    processing_class=mt5_tokenizer
)

test_start = time.time()
test_output = test_trainer.train()

print("\nTest duration:", round(time.time() - test_start, 2), "seconds")
print("Test training loss:", test_output.training_loss)
print("Finite loss:", torch.isfinite(
    torch.tensor(test_output.training_loss)
).item())
print("Final training logs:", test_trainer.state.log_history[-5:])

Step,Training Loss
1,94.616318
2,58.776867
3,46.756012
4,49.038528
5,53.754494



Test duration: 2.73 seconds
Test training loss: 60.588443756103516
Finite loss: True
Final training logs: [{'loss': 58.7768669128418, 'grad_norm': 15115.734375, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.0002, 'step': 2}, {'loss': 46.756011962890625, 'grad_norm': 9342.9130859375, 'learning_rate': 6e-06, 'epoch': 0.0003, 'step': 3}, {'loss': 49.03852844238281, 'grad_norm': 6589.7783203125, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.0004, 'step': 4}, {'loss': 53.754493713378906, 'grad_norm': 12454.0390625, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.0005, 'step': 5}, {'train_runtime': 1.8842, 'train_samples_per_second': 10.615, 'train_steps_per_second': 2.654, 'total_flos': 743554252800.0, 'train_loss': 60.588443756103516, 'epoch': 0.0005, 'step': 5}]


In [ ]:
import torch

mt5_model.eval()

batch = data_collator([mt5_train[i] for i in range(4)])
batch = {
    key: value.to(device)
    for key, value in batch.items()
    if isinstance(value, torch.Tensor)
}

with torch.no_grad():
    outputs = mt5_model(**batch)

labels = batch["labels"]
valid_labels = labels[labels != -100]

print("Loss:", outputs.loss.item())
print("Logits shape:", tuple(outputs.logits.shape))
print("Logits finite:", torch.isfinite(outputs.logits).all().item())
print("Label range:", int(valid_labels.min()), "to", int(valid_labels.max()))
print("Tokenizer vocabulary size:", len(mt5_tokenizer))
print("Number of ignored padding labels:", int((labels == -100).sum()))
print("Decoder start token:", mt5_model.config.decoder_start_token_id)
print("Pad token:", mt5_tokenizer.pad_token_id)

Loss: 14.594717979431152
Logits shape: (4, 72, 250112)
Logits finite: True
Label range: 1 to 182244
Tokenizer vocabulary size: 250100
Number of ignored padding labels: 174
Decoder start token: 0
Pad token: 0


In [ ]:
import torch

mt5_model.eval()

test_sentences = [
    "Hello, how are you?",
    "India is a beautiful country.",
    "What is your name?"
]

for sentence in test_sentences:
    inputs = mt5_tokenizer(
        "translate English to Hindi: " + sentence,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).to(device)

    with torch.no_grad():
        generated = mt5_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False
        )

    translation = mt5_tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )

    print("English:", sentence)
    print("Hindi:", translation)
    print()

English: Hello, how are you?
Hindi: <extra_id_0>

English: India is a beautiful country.
Hindi: <extra_id_0>

English: What is your name?
Hindi: <extra_id_0>



In [ ]:
print("Tokenizer class:", type(mt5_tokenizer).__name__)
print("Model class:", type(mt5_model).__name__)
print("Model name:", mt5_model.config._name_or_path)
print("Decoder start token:", mt5_model.config.decoder_start_token_id)
print("EOS token:", mt5_tokenizer.eos_token, mt5_tokenizer.eos_token_id)
print("PAD token:", mt5_tokenizer.pad_token, mt5_tokenizer.pad_token_id)
print("Sentinel token ID:", mt5_tokenizer.convert_tokens_to_ids("<extra_id_0>"))

print("\nTesting different prompts:")
for prompt in [
    "translate English to Hindi: Hello, how are you?",
    "Hello, how are you?",
    "translate English to Hindi: India is a beautiful country."
]:
    inputs = mt5_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=128
    ).to(device)

    with torch.no_grad():
        output = mt5_model.generate(
            **inputs,
            max_new_tokens=40,
            num_beams=1,
            do_sample=False,
            decoder_start_token_id=mt5_model.config.decoder_start_token_id,
            eos_token_id=mt5_tokenizer.eos_token_id,
            pad_token_id=mt5_tokenizer.pad_token_id
        )

    print(repr(prompt), "->", repr(mt5_tokenizer.decode(
        output[0], skip_special_tokens=False
    )))

Tokenizer class: T5Tokenizer
Model class: MT5ForConditionalGeneration
Model name: google/mt5-small
Decoder start token: 0
EOS token: </s> 1
PAD token: <pad> 0
Sentinel token ID: 2

Testing different prompts:
'translate English to Hindi: Hello, how are you?' -> '<pad> <extra_id_0></s>'
'Hello, how are you?' -> '<pad> <extra_id_0></s>'
'translate English to Hindi: India is a beautiful country.' -> '<pad> <extra_id_0></s>'


In [ ]:
import torch

shared = mt5_model.shared.weight
lm_head = mt5_model.lm_head.weight

print("Shared embedding shape:", tuple(shared.shape))
print("LM head shape:", tuple(lm_head.shape))
print("Same underlying storage:", shared.data_ptr() == lm_head.data_ptr())
print("Shared embedding mean:", shared.float().mean().item())
print("LM head mean:", lm_head.float().mean().item())
print("Mean absolute difference:", (shared.float() - lm_head.float()).abs().mean().item())

# Check whether the model predicts the sentinel token immediately
mt5_model.eval()
inputs = mt5_tokenizer(
    "translate English to Hindi: Hello, how are you?",
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output = mt5_model.generate(**inputs, max_new_tokens=5)

print("Generated token IDs:", output[0].tolist())
print("Generated tokens:", mt5_tokenizer.convert_ids_to_tokens(output[0].tolist()))

Shared embedding shape: (250112, 512)
LM head shape: (250112, 512)
Same underlying storage: False
Shared embedding mean: 0.07742290198802948
LM head mean: 0.00415207352489233
Mean absolute difference: 11.467988967895508
Generated token IDs: [0, 250099, 1]
Generated tokens: ['<pad>', '▁<extra_id_0>', '</s>']


In [ ]:
import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM

del mt5_model
torch.cuda.empty_cache()

config = AutoConfig.from_pretrained("google/mt5-small")
config.tie_word_embeddings = False

diagnostic_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/mt5-small",
    config=config,
    dtype=torch.float32
).to(device)

diagnostic_model.eval()

sentence = "Hello, how are you?"
inputs = mt5_tokenizer(
    "translate English to Hindi: " + sentence,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    generated = diagnostic_model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=4
    )

print("Parameters:", f"{sum(p.numel() for p in diagnostic_model.parameters()):,}")
print("Output:", repr(mt5_tokenizer.decode(
    generated[0], skip_special_tokens=True
)))

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

Parameters: 556,291,456
Output: '<extra_id_0>'


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"

# Reload the original checkpoint cleanly
del diagnostic_model
torch.cuda.empty_cache()

mt5_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32
).to(device)

# Use the task prefix and generate
text = "Hello, how are you?"
inputs = mt5_tokenizer(
    "translate English to Hindi: " + text,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(device)

mt5_model.eval()
with torch.no_grad():
    output = mt5_model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=4,
        repetition_penalty=1.2
    )

print("Translation:", mt5_tokenizer.decode(
    output[0], skip_special_tokens=True
))

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Translation: <extra_id_0>


In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

del mt5_model
if "diagnostic_model" in globals():
    del diagnostic_model
if "test_trainer" in globals():
    del test_trainer
if "trainer" in globals():
    del trainer
torch.cuda.empty_cache()

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"

opus_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
opus_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
opus_model.eval()

examples = [
    "Hello, how are you?",
    "India is a beautiful country.",
    "What is your name?"
]

for sentence in examples:
    inputs = opus_tokenizer(
        sentence, return_tensors="pt",
        truncation=True, max_length=128
    ).to(device)

    with torch.no_grad():
        output = opus_model.generate(**inputs, max_new_tokens=64)

    print("EN:", sentence)
    print("HI:", opus_tokenizer.decode(
        output[0], skip_special_tokens=True
    ))
    print()

print("Parameters:", f"{sum(p.numel() for p in opus_model.parameters()):,}")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (6 outputs left out)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: Hello, how are you?
HI: हैलो, तुम कैसे हो?



[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: India is a beautiful country.
HI: भारत एक खूबसूरत देश है.

EN: What is your name?
HI: आपका Windows Live कूटशब्द क्या है?

Parameters: 76,381,184


In [ ]:
import time
import torch
import sacrebleu
from tqdm.auto import tqdm

opus_model.eval()

test_en = [row["en"] for row in test_ds]
test_hi = [row["hi"] for row in test_ds]

opus_predictions = []
opus_start = time.time()

for sentence in tqdm(test_en, desc="Evaluating OPUS-MT"):
    inputs = opus_tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        output = opus_model.generate(
            **inputs,
            max_new_tokens=128
        )

    opus_predictions.append(
        opus_tokenizer.decode(output[0], skip_special_tokens=True)
    )

opus_total_time = time.time() - opus_start

opus_bleu = sacrebleu.corpus_bleu(
    opus_predictions, [test_hi]
).score

opus_chrf = sacrebleu.corpus_chrf(
    opus_predictions, [test_hi]
).score

opus_parameters = sum(p.numel() for p in opus_model.parameters())

print("\n========== OPUS-MT RESULTS ==========")
print("Test examples:", len(test_en))
print(f"BLEU: {opus_bleu:.2f}")
print(f"chrF: {opus_chrf:.2f}")
print(f"Total inference time: {opus_total_time:.2f} seconds")
print(f"Average inference time: {opus_total_time / len(test_en):.4f} seconds/sentence")
print(f"Parameters: {opus_parameters:,}")
print(f"Total inference time in minutes: {opus_total_time / 60:.2f}")

print("\nSample predictions:")
for i in range(5):
    print(f"\nEN: {test_en[i]}")
    print(f"Reference: {test_hi[i]}")
    print(f"Predicted: {opus_predictions[i]}")

In [ ]:
print("========== OPUS-MT RESULTS ==========")
print("Test examples:", len(test_en))
print(f"BLEU: {opus_bleu:.2f}")
print(f"chrF: {opus_chrf:.2f}")
print(f"Total inference time: {opus_total_time:.2f} seconds")
print(f"Average inference time: {opus_total_time / len(test_en):.4f} seconds/sentence")
print(f"Parameters: {sum(p.numel() for p in opus_model.parameters()):,}")

print("\nSample predictions:")
for i in range(5):
    print(f"\nEN: {test_en[i]}")
    print(f"Reference: {test_hi[i]}")
    print(f"Predicted: {opus_predictions[i]}")

========== OPUS-MT RESULTS ==========
Test examples: 2507
BLEU: 9.88
chrF: 32.33
Total inference time: 739.94 seconds
Average inference time: 0.2952 seconds/sentence
Parameters: 76,381,184

Sample predictions:

EN: A black box in your car?
Reference: आपकी कार में ब्लैक बॉक्स?
Predicted: अपनी कार में एक ब्लैक बॉक्स?

EN: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Predicted: के रूप में अमेरिका के सड़क योजनार एक टूटी हुई सड़क प्रणाली को सुधारने के लिए नकदी को खोजने के लिए संघर्ष कर रहे हैं, कई एक छोटे से काले बॉक्स में एक समाधान देखने के लिए शुरू कर रहे हैं कि अपनी कार के माध्यम से फिट.

EN: The devices, which 

In [ ]:
import json
import pandas as pd

opus_results = {
    "model": "Helsinki-NLP/opus-mt-en-hi",
    "test_examples": len(test_en),
    "bleu": opus_bleu,
    "chrf": opus_chrf,
    "total_inference_seconds": opus_total_time,
    "average_inference_seconds": opus_total_time / len(test_en),
    "parameters": sum(p.numel() for p in opus_model.parameters()),
    "training_time_seconds": None
}

with open("opus_mt_results.json", "w", encoding="utf-8") as f:
    json.dump(opus_results, f, indent=2, ensure_ascii=False)

pd.DataFrame({
    "English": test_en,
    "Reference Hindi": test_hi,
    "Predicted Hindi": opus_predictions
}).to_csv("opus_mt_predictions.csv", index=False, encoding="utf-8-sig")

print("Saved opus_mt_results.json")
print("Saved opus_mt_predictions.csv")
print(json.dumps(opus_results, indent=2))

Saved opus_mt_results.json
Saved opus_mt_predictions.csv
{
  "model": "Helsinki-NLP/opus-mt-en-hi",
  "test_examples": 2507,
  "bleu": 9.88482404445027,
  "chrf": 32.32542406669296,
  "total_inference_seconds": 739.9426021575928,
  "average_inference_seconds": 0.2951506191294746,
  "parameters": 76381184,
  "training_time_seconds": null
}


In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

LLAMA_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_NAME)
llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

print("Model:", LLAMA_NAME)
print("Parameters:", f"{sum(p.numel() for p in llama_model.parameters()):,}")
print("Device:", next(llama_model.parameters()).device)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 1,100,048,384
Device: cuda:0


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an English to Hindi translation assistant. Translate accurately. Output only the Hindi translation."
    },
    {
        "role": "user",
        "content": "Translate into Hindi: A black box in your car?"
    }
]

prompt = llama_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = llama_tokenizer(
    prompt,
    return_tensors="pt"
).to(llama_model.device)

with torch.no_grad():
    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id
    )

translation = llama_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("English:", "A black box in your car?")
print("Hindi prediction:", translation)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English: A black box in your car?
Hindi prediction: एक रूसी का एक उपकरण है उसके साथ से स्टेटस का उपकरण का उसका उपकरण ह


In [ ]:
messages = [
    {
        "role": "system",
        "content": "Translate English sentences into natural Hindi. Do not explain. Return only the translated sentence in Devanagari script."
    },
    {
        "role": "user",
        "content": "English sentence: Hello, how are you?\nHindi translation:"
    }
]

prompt = llama_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)

with torch.no_grad():
    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=llama_tokenizer.eos_token_id
    )

print(llama_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip())

[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


हैदराबाद के समय की सुनिश्चित हैं आप?

English sentence


In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME)

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model:", QWEN_NAME)
print("Parameters:", f"{sum(p.numel() for p in qwen_model.parameters()):,}")
print("Device:", next(qwen_model.parameters()).device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494,032,768
Device: cuda:0


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an English-to-Hindi translator. Translate accurately into natural Hindi using Devanagari script. Output only the translation."
    },
    {
        "role": "user",
        "content": "Translate into Hindi: A black box in your car?"
    }
]

prompt = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(
    prompt,
    return_tensors="pt"
).to(qwen_model.device)

start = time.time()

with torch.no_grad():
    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=qwen_tokenizer.eos_token_id
    )

qwen_test_time = time.time() - start

qwen_translation = qwen_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("English:", "A black box in your car?")
print("Hindi prediction:", qwen_translation)
print(f"Inference time: {qwen_test_time:.3f} seconds")

English: A black box in your car?
Hindi prediction: एक बाजार को भूमिका में है?
Inference time: 1.860 seconds


In [ ]:
import time
import pandas as pd
import sacrebleu
import torch

N_EVAL = 100
llama_rows = test_ds.select(range(N_EVAL))
llama_predictions = []
llama_references = []

start_time = time.time()

for i in range(N_EVAL):
    english = llama_rows[i]["en"]
    hindi = llama_rows[i]["hi"]

    messages = [
        {
            "role": "system",
            "content": "Translate English to natural Hindi in Devanagari script. Output only the translation."
        },
        {
            "role": "user",
            "content": f"Translate into Hindi:\n{english}"
        }
    ]

    prompt = llama_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llama_tokenizer(
        prompt, return_tensors="pt"
    ).to(llama_model.device)

    with torch.no_grad():
        output = llama_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=llama_tokenizer.eos_token_id
        )

    prediction = llama_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    llama_predictions.append(prediction)
    llama_references.append(hindi)

llama_elapsed = time.time() - start_time
llama_bleu = sacrebleu.corpus_bleu(
    llama_predictions, [llama_references]
).score
llama_chrf = sacrebleu.corpus_chrf(
    llama_predictions, [llama_references]
).score

llama_results = {
    "model": "TinyLlama-1.1B-Chat (zero-shot prompting)",
    "parameters": sum(p.numel() for p in llama_model.parameters()),
    "test_examples": N_EVAL,
    "BLEU": round(llama_bleu, 2),
    "chrF": round(llama_chrf, 2),
    "total_inference_seconds": round(llama_elapsed, 2),
    "average_seconds_per_sentence": round(llama_elapsed / N_EVAL, 3)
}

print(pd.Series(llama_results).to_string())
print("\nSample comparisons:")
for i in range(min(5, N_EVAL)):
    print(f"\nEN:   {llama_rows[i]['en']}")
    print(f"REF:  {llama_references[i]}")
    print(f"PRED: {llama_predictions[i]}")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

model                           TinyLlama-1.1B-Chat (zero-shot prompting)
parameters                                                     1100048384
test_examples                                                         100
BLEU                                                                 0.63
chrF                                                                13.25
total_inference_seconds                                            307.98
average_seconds_per_sentence                                         3.08

Sample comparisons:

EN:   A black box in your car?
REF:  आपकी कार में ब्लैक बॉक्स?
PRED: एक रूस का घर से संबंधित काफीर है उसका उपयोग करने के लिए किसी भी स्थान के साथ संबंधित है।

EN:   As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
REF:  जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ

In [ ]:
try:
    import peft
    print("PEFT installed:", peft.__version__)
except ImportError:
    print("PEFT is not installed")

PEFT installed: 0.21.2


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load model only if it is not already in memory
if "model" not in globals() or not isinstance(model, torch.nn.Module):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Configure LoRA: train only a small number of adapter parameters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("GPU:", torch.cuda.get_device_name(0))

NameError: name 'tokenizer' is not defined

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer and model explicitly
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Configure LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("GPU:", torch.cuda.get_device_name(0))
print("Tokenizer and LoRA setup completed.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
GPU: Tesla T4
Tokenizer and LoRA setup completed.


In [ ]:
%pip install -q "torchao>0.16.0"

In [ ]:
import torch
from datasets import Dataset

# Use a small subset for the LoRA experiment
LORA_TRAIN_SIZE = min(2000, len(train_ds))
lora_train_raw = train_ds.shuffle(seed=42).select(range(LORA_TRAIN_SIZE))

MAX_LENGTH = 160

def format_example(example):
    prompt = (
        "Translate the following English sentence into Hindi.\n"
        f"English: {example['en']}\n"
        "Hindi:"
    )
    answer = " " + example["hi"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"]

    answer_ids = tokenizer(
        answer, add_special_tokens=False
    )["input_ids"]

    # Reserve space for the answer; discard overly long examples
    prompt_ids = prompt_ids[:MAX_LENGTH - 2]
    answer_ids = answer_ids[:MAX_LENGTH - len(prompt_ids)]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

lora_train = lora_train_raw.map(
    format_example,
    remove_columns=lora_train_raw.column_names
)

# Remove examples with no target tokens
lora_train = lora_train.filter(
    lambda x: any(token != -100 for token in x["labels"])
)

print("Training examples:", len(lora_train))
print("Sequence length example:", len(lora_train[0]["input_ids"]))
print("Prepared for LoRA fine-tuning.")

NameError: name 'train_ds' is not defined

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)
print("Available splits:", list(dataset.keys()))
print("Training columns:", dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})
Available splits: ['train', 'validation', 'test']
Training columns: ['translation']


In [ ]:
import re
import unicodedata
from datasets import Dataset

def clean_text(text):
    text = unicodedata.normalize("NFC", str(text))
    return re.sub(r"\s+", " ", text).strip()

# Extract English-Hindi pairs
pairs = []
seen = set()

for row in dataset["train"]:
    en = clean_text(row["translation"]["en"])
    hi = clean_text(row["translation"]["hi"])

    if not en or not hi:
        continue

    key = (en, hi)
    if key not in seen:
        seen.add(key)
        pairs.append({"en": en, "hi": hi})

train_ds = Dataset.from_list(pairs)

# Reproduce the earlier 40,000-example training sample
train_ds = train_ds.shuffle(seed=42).select(
    range(min(40000, len(train_ds)))
)

print("Training examples available:", len(train_ds))
print(train_ds[0])

Training examples available: 40000
{'en': 'I promise never to do it again, I said.', 'hi': 'मैं आपसे वायदा करता हूं कि फिर कभी ऐसा नहीं करूंगा, मैने कहा।'}


In [ ]:
import torch
from datasets import Dataset

LORA_TRAIN_SIZE = 2000
lora_train_raw = train_ds.shuffle(seed=42).select(
    range(min(LORA_TRAIN_SIZE, len(train_ds)))
)

MAX_LENGTH = 160

def format_example(example):
    prompt = (
        "Translate the following English sentence into Hindi.\n"
        f"English: {example['en']}\n"
        "Hindi:"
    )
    answer = " " + example["hi"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"]
    answer_ids = tokenizer(
        answer, add_special_tokens=False
    )["input_ids"]

    # Keep the sequence within the model's context limit
    prompt_ids = prompt_ids[:MAX_LENGTH - 2]
    answer_ids = answer_ids[:MAX_LENGTH - len(prompt_ids)]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

lora_train = lora_train_raw.map(
    format_example,
    remove_columns=lora_train_raw.column_names
)

lora_train = lora_train.filter(
    lambda x: any(token != -100 for token in x["labels"])
)

print("LoRA training examples:", len(lora_train))
print("Example sequence length:", len(lora_train[0]["input_ids"]))
print("Dataset preparation complete.")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

LoRA training examples: 2000
Example sequence length: 160
Dataset preparation complete.


In [ ]:
import torch
import time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Pad batches dynamically; prompt labels (-100) remain ignored in the loss
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir="./tinyllama_en_hi_lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lora_train,
    data_collator=data_collator
)

start_time = time.time()
train_result = trainer.train()
lora_training_seconds = time.time() - start_time

print(f"\nLoRA training time: {lora_training_seconds / 60:.2f} minutes")
print(f"Training loss: {train_result.training_loss:.4f}")
model.save_pretrained("./tinyllama_en_hi_lora/final_adapter")
tokenizer.save_pretrained("./tinyllama_en_hi_lora/final_adapter")
print("LoRA adapter saved.")

Step,Training Loss
25,1.310789
50,1.265767
75,1.235007
100,1.219922
125,1.199981
150,1.171387
175,1.179558
200,1.199828
225,1.183846
250,1.203459



LoRA training time: 4.20 minutes
Training loss: 1.2170
LoRA adapter saved.


In [ ]:
import torch
import time
import pandas as pd
import unicodedata
from datasets import load_dataset
from sacrebleu.metrics import BLEU, CHRF

# Load the official test split
test_data = load_dataset(
    "cfilt/iitb-english-hindi",
    split="test"
)

def clean_text(text):
    return " ".join(
        unicodedata.normalize("NFC", str(text)).split()
    )

test_pairs = []
for row in test_data:
    en = clean_text(row["translation"]["en"])
    hi = clean_text(row["translation"]["hi"])
    if en and hi:
        test_pairs.append({"en": en, "hi": hi})

# Keep the same 100-example evaluation size
eval_pairs = test_pairs[:100]

model.eval()
predictions = []
references = []
inference_times = []

for i, pair in enumerate(eval_pairs):
    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following English sentence into Hindi.\n"
                f"English: {pair['en']}\n"
                "Hindi:"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.time()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_times.append(time.time() - start)

    # Decode only newly generated tokens
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    predictions.append(prediction)
    references.append(pair["hi"])

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/{len(eval_pairs)}")

# Calculate evaluation metrics
bleu_score = BLEU().corpus_score(
    predictions, [references]
).score

chrf_score = CHRF().corpus_score(
    predictions, [references]
).score

total_inference = sum(inference_times)
avg_inference = total_inference / len(inference_times)

print("\n===== TINYLLAMA LoRA RESULTS =====")
print(f"Test examples: {len(predictions)}")
print(f"BLEU: {bleu_score:.2f}")
print(f"chrF: {chrf_score:.2f}")
print(f"Total inference time: {total_inference:.2f} seconds")
print(f"Average inference time: {avg_inference:.2f} seconds/sentence")
print(f"Training time: {lora_training_seconds:.2f} seconds")

# Save predictions for the report
results_df = pd.DataFrame({
    "English": [p["en"] for p in eval_pairs],
    "Reference Hindi": references,
    "Predicted Hindi": predictions
})
results_df.to_csv("tinyllama_lora_predictions.csv", index=False)

print("\nSample predictions:")
display(results_df.head(10))
print("\nSaved: tinyllama_lora_predictions.csv")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Evaluated 100/100

===== TINYLLAMA LoRA RESULTS =====
Test examples: 100
BLEU: 0.66
chrF: 12.99
Total inference time: 323.70 seconds
Average inference time: 3.24 seconds/sentence
Training time: 251.94 seconds

Sample predictions:


,English,Reference Hindi,Predicted Hindi
0,A black box in your car?,आपकी कार में ब्लैक बॉक्स?,कार में का एक ब्राउज़र है?
1,As America's road planners struggle to find th...,"जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए...",अमेरिका के एक एक्सप्रेस के राष्ट्रपति के साथ ए...
2,"The devices, which track every mile a motorist...","यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्...",विदेश के दौरान इस क्षेत्र में किसी के लिए एक ए...
3,The usually dull arena of highway planning has...,आम तौर पर हाईवे नियोजन जैसा उबाऊ काम भी अचानक ...,उत्तर प्रदेश के एक एक्सप्रेस के साथ स्थानीय एक...
4,Libertarians have joined environmental groups ...,"आपने द्वारा ड्राइव किए गए मील, तथा संभवतः ड्रा...",लॉबिंग से संबंधित संस्थाओं ने सरकार के लिए एक ...
5,The tea party is aghast.,चाय पार्टी भौचक्की है।,इस समझ के लिए तेज़ पर है।
6,The American Civil Liberties Union is deeply c...,अमेरिकी नागरिक स्वतंत्रता संघ भी विभिन्न प्रका...,अमेरिकी अनुराग संग्राम के संग्राम के संग्राम क...
7,And while Congress can't agree on whether to p...,जबकि कांग्रेस इस बात पर सहमत नहीं हो सकी कि आग...,और किसी के साथ कोरोना के लिए कांग्रेस ने कहा क...
8,"They are exploring how, over the next decade, ...",वे यह खोज कर रहे हैं कि अगले दशक में वे किस तर...,उन्होंने कहा कि उन्हें एक स्थान पर किसी को एक ...
9,Thousands of motorists have already taken the ...,हजारों मोटर-चालकों ने टेस्ट ड्राइव के लिए पहले...,(अंग्रेज़ी में) अब किसी को एक साल के लिए एक सा...



Saved: tinyllama_lora_predictions.csv


In [ ]:
import json
import pandas as pd

results = [
    {"model": "Transformer from scratch", "n": 2507, "params": 20238817, "BLEU": 0.15, "chrF": 4.72, "training_min": 3.85},
    {"model": "mT5-small", "n": None, "params": 300176768, "BLEU": None, "chrF": None, "training_min": None},
    {"model": "OPUS-MT en-hi (additional baseline)", "n": 2507, "params": 76381184, "BLEU": 9.88, "chrF": 32.33, "training_min": None},
    {"model": "TinyLlama zero-shot", "n": 100, "params": 1100048384, "BLEU": 0.63, "chrF": 13.25, "training_min": 0},
    {"model": "TinyLlama LoRA", "n": 100, "params": 1101174784, "BLEU": 0.66, "chrF": 12.99, "training_min": 4.20},
]

pd.DataFrame(results).to_csv("experiment_results.csv", index=False)

with open("experiment_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved experiment_results.csv and experiment_results.json")

Saved experiment_results.csv and experiment_results.json


In [ ]:
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

MODEL_NAME = "google/gemma-2-2b-it"

gemma_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

gemma_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Gemma loaded successfully!")
print("Parameters:", sum(p.numel() for p in gemma_model.parameters()))
print("GPU:", torch.cuda.get_device_name(0))

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

print("Hugging Face authentication successful!")

Hugging Face authentication successful!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-2-2b-it"

gemma_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

gemma_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

gemma_model.eval()

gemma_params = sum(
    p.numel() for p in gemma_model.parameters()
)

print("Gemma loaded successfully!")
print(f"Total parameters: {gemma_params:,}")
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma loaded successfully!
Total parameters: 2,614,341,888
GPU: Tesla T4
GPU memory allocated: 6.95 GB


In [ ]:
import torch

def gemma_translate(text):
    messages = [
        {
            "role": "user",
            "content": (
                "Translate this English sentence into natural Hindi. "
                "Return only the Hindi translation.\n\n"
                f"English: {text}"
            )
        }
    ]

    prompt = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = gemma_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(gemma_model.device)

    with torch.inference_mode():
        output = gemma_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return gemma_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

test_sentences = [
    "A black box in your car?",
    "Hello, how are you?",
    "India is a diverse country with many languages."
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Hindi:", gemma_translate(sentence))
    print("-" * 60)

English: A black box in your car?
Hindi: कार में काले बॉक्स है?
------------------------------------------------------------
English: Hello, how are you?
Hindi: नमस्ते, आप कैसे हैं?
------------------------------------------------------------
English: India is a diverse country with many languages.
Hindi: भारत एक विविध देश है, जहाँ कई भाषाएँ हैं।
------------------------------------------------------------


In [ ]:
import time
import pandas as pd
import torch
from datasets import load_dataset
from sacrebleu.metrics import BLEU, CHRF
import unicodedata

# Load official test data
gemma_test_data = load_dataset(
    "cfilt/iitb-english-hindi",
    split="test"
)

def normalize_text(text):
    return " ".join(
        unicodedata.normalize("NFC", str(text)).split()
    )

gemma_eval_pairs = []
for row in gemma_test_data:
    en = normalize_text(row["translation"]["en"])
    hi = normalize_text(row["translation"]["hi"])
    if en and hi:
        gemma_eval_pairs.append({"en": en, "hi": hi})

# Same evaluation size as TinyLlama
gemma_eval_pairs = gemma_eval_pairs[:100]

gemma_predictions = []
gemma_references = []
gemma_inference_times = []

gemma_model.eval()

for i, pair in enumerate(gemma_eval_pairs):
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.time()
    prediction = gemma_translate(pair["en"])

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    gemma_inference_times.append(time.time() - start)
    gemma_predictions.append(prediction)
    gemma_references.append(pair["hi"])

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/100")

gemma_bleu = BLEU().corpus_score(
    gemma_predictions, [gemma_references]
).score

gemma_chrf = CHRF().corpus_score(
    gemma_predictions, [gemma_references]
).score

gemma_total_time = sum(gemma_inference_times)
gemma_avg_time = gemma_total_time / len(gemma_inference_times)

gemma_results_df = pd.DataFrame({
    "English": [p["en"] for p in gemma_eval_pairs],
    "Reference Hindi": gemma_references,
    "Predicted Hindi": gemma_predictions
})

print("\n===== GEMMA ZERO-SHOT RESULTS =====")
print("Evaluation examples:", len(gemma_predictions))
print("Parameters:", f"{gemma_params:,}")
print(f"BLEU: {gemma_bleu:.2f}")
print(f"chrF: {gemma_chrf:.2f}")
print(f"Total inference time: {gemma_total_time:.2f} seconds")
print(f"Average inference time: {gemma_avg_time:.2f} seconds/sentence")

gemma_results_df.to_csv(
    "gemma_zero_shot_predictions.csv", index=False
)

display(gemma_results_df.head(10))

Evaluated 10/100
Evaluated 20/100
Evaluated 30/100
Evaluated 40/100
Evaluated 50/100
Evaluated 60/100
Evaluated 70/100
Evaluated 80/100
Evaluated 90/100
Evaluated 100/100

===== GEMMA ZERO-SHOT RESULTS =====
Evaluation examples: 100
Parameters: 2,614,341,888
BLEU: 12.43
chrF: 38.84
Total inference time: 197.30 seconds
Average inference time: 1.97 seconds/sentence


,English,Reference Hindi,Predicted Hindi
0,A black box in your car?,आपकी कार में ब्लैक बॉक्स?,कार में काले बॉक्स है?
1,As America's road planners struggle to find th...,"जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए...",अमेरिका के रोड प्लानर अपनी सड़क प्रणाली को मरम...
2,"The devices, which track every mile a motorist...","यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्...",वाहन जो कि हर मील की दूरी ट्रैक करते हैं और उस...
3,The usually dull arena of highway planning has...,आम तौर पर हाईवे नियोजन जैसा उबाऊ काम भी अचानक ...,हाईवेय योजना का आम तौर पर उदासीन क्षेत्र ने अच...
4,Libertarians have joined environmental groups ...,"आपने द्वारा ड्राइव किए गए मील, तथा संभवतः ड्रा...",लैबरिस्ट्स ने पर्यावरण समूहों के साथ मिलकर सरक...
5,The tea party is aghast.,चाय पार्टी भौचक्की है।,chai ka party आहत है।
6,The American Civil Liberties Union is deeply c...,अमेरिकी नागरिक स्वतंत्रता संघ भी विभिन्न प्रका...,अमेरिकन सिविल लाइबर्टीज यूनीयॉन्स को गहराई से ...
7,And while Congress can't agree on whether to p...,जबकि कांग्रेस इस बात पर सहमत नहीं हो सकी कि आग...,"जबकि कांग्रेस को आगे बढ़ने पर सहमति नहीं है, क..."
8,"They are exploring how, over the next decade, ...",वे यह खोज कर रहे हैं कि अगले दशक में वे किस तर...,"वे अगले दशक में, ड्राइवरों को सड़क पर रोल करने..."
9,Thousands of motorists have already taken the ...,हजारों मोटर-चालकों ने टेस्ट ड्राइव के लिए पहले...,हजारों वाहन चालक पहले ही ब्लैक बॉक्सों को ले च...


In [ ]:
import torch
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA for Gemma 2
gemma_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

gemma_model = get_peft_model(
    gemma_model,
    gemma_lora_config
)

gemma_model.print_trainable_parameters()

print("Gemma LoRA adapters attached.")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221
Gemma LoRA adapters attached.
GPU: Tesla T4
GPU memory allocated: 6.96 GB


In [ ]:
import torch
import time
from transformers import Trainer, TrainingArguments
from datasets import Dataset

# Reuse the same 2,000 English-Hindi pairs
gemma_train_raw = train_ds.shuffle(seed=42).select(range(2000))

MAX_LENGTH = 160

def format_gemma_example(example):
    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following English sentence into natural Hindi. "
                "Return only the Hindi translation.\n\n"
                f"English: {example['en']}"
            )
        },
        {
            "role": "assistant",
            "content": example["hi"]
        }
    ]

    # Tokenize the complete chat example using Gemma's native template
    tokenized = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )

    input_ids = tokenized["input_ids"][:MAX_LENGTH]

    # Train on the complete formatted sequence
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": input_ids.copy()
    }

gemma_train = gemma_train_raw.map(
    format_gemma_example,
    remove_columns=gemma_train_raw.column_names
)

print("Gemma training examples:", len(gemma_train))
print("Example sequence length:", len(gemma_train[0]["input_ids"]))
print("Gemma training data ready.")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Gemma training examples: 2000
Example sequence length: 160
Gemma training data ready.


In [ ]:
import time
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Use EOS as padding if the tokenizer has no pad token
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

gemma_model.config.pad_token_id = gemma_tokenizer.pad_token_id
gemma_model.config.use_cache = False

# Pad each batch dynamically and ignore padding in the loss
data_collator = DataCollatorForSeq2Seq(
    tokenizer=gemma_tokenizer,
    model=gemma_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

training_args = TrainingArguments(
    output_dir="./gemma_lora_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    fp16=True,
    optim="adamw_torch",
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=True
)

trainer = Trainer(
    model=gemma_model,
    args=training_args,
    train_dataset=gemma_train,
    data_collator=data_collator
)

start_time = time.time()
train_output = trainer.train()
gemma_lora_training_seconds = time.time() - start_time

print("\nGemma LoRA training complete!")
print(f"Training time: {gemma_lora_training_seconds / 60:.2f} minutes")
print(f"Final training loss: {train_output.training_loss:.4f}")

trainable = sum(p.numel() for p in gemma_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in gemma_model.parameters())

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.4f}%")

Step,Training Loss
25,4.086603
50,2.636286
75,2.155222
100,1.994085
125,1.904527
150,1.791203
175,1.845134
200,1.869052
225,1.797104
250,1.868384



Gemma LoRA training complete!
Training time: 14.48 minutes
Final training loss: 2.1948
Trainable parameters: 3,194,880
Total parameters: 2,617,536,768
Trainable percentage: 0.1221%


In [ ]:
import time
import pandas as pd
import torch
from sacrebleu.metrics import BLEU, CHRF

gemma_model.eval()
gemma_model.config.use_cache = True

# Use the same first 100 official test examples as zero-shot evaluation
gemma_eval_data = test_ds.select(range(100))

gemma_lora_predictions = []
gemma_references = []
gemma_inference_start = time.time()

for i, example in enumerate(gemma_eval_data):
    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following English sentence into natural Hindi. "
                "Return only the Hindi translation.\n\n"
                f"English: {example['en']}"
            )
        }
    ]

    inputs = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(gemma_model.device)

    with torch.inference_mode():
        outputs = gemma_model.generate(
            inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id,
            eos_token_id=gemma_tokenizer.eos_token_id
        )

    # Decode only newly generated tokens
    new_tokens = outputs[0][inputs.shape[-1]:]
    prediction = gemma_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    gemma_lora_predictions.append(prediction)
    gemma_references.append(example["hi"])

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/100 examples")

gemma_lora_inference_seconds = time.time() - gemma_inference_start

gemma_lora_bleu = BLEU().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_chrf = CHRF().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_results = {
    "model": "Gemma-2-2B-IT LoRA",
    "test_examples": len(gemma_lora_predictions),
    "bleu": gemma_lora_bleu,
    "chrf": gemma_lora_chrf,
    "total_inference_seconds": gemma_lora_inference_seconds,
    "average_inference_seconds": (
        gemma_lora_inference_seconds / len(gemma_lora_predictions)
    ),
    "training_seconds": gemma_lora_training_seconds,
    "trainable_parameters": sum(
        p.numel() for p in gemma_model.parameters() if p.requires_grad
    ),
    "total_parameters": sum(
        p.numel() for p in gemma_model.parameters()
    )
}

pd.DataFrame({
    "English": [x["en"] for x in gemma_eval_data],
    "Reference_Hindi": gemma_references,
    "Gemma_LoRA_Hindi": gemma_lora_predictions
}).to_csv("gemma_lora_predictions.csv", index=False)

pd.DataFrame([gemma_lora_results]).to_csv(
    "gemma_lora_results.csv", index=False
)

print("\n--- GEMMA LoRA RESULTS ---")
for key, value in gemma_lora_results.items():
    print(f"{key}: {value}")

print("\n--- SAMPLE PREDICTIONS ---")
for i in range(min(5, len(gemma_lora_predictions))):
    print(f"\nEnglish: {gemma_eval_data[i]['en']}")
    print(f"Reference: {gemma_references[i]}")
    print(f"Gemma LoRA: {gemma_lora_predictions[i]}")

NameError: name 'test_ds' is not defined

In [ ]:
from datasets import load_dataset
import unicodedata

# Reload the original dataset
dataset = load_dataset("cfilt/iitb-english-hindi")

def clean_text(text):
    return " ".join(
        unicodedata.normalize("NFC", text).strip().split()
    )

def clean_pair(example):
    return {
        "en": clean_text(example["translation"]["en"]),
        "hi": clean_text(example["translation"]["hi"])
    }

test_ds = dataset["test"].map(
    clean_pair,
    remove_columns=dataset["test"].column_names
)

test_ds = test_ds.filter(
    lambda x: bool(x["en"]) and bool(x["hi"])
)

print("Official test examples available:", len(test_ds))
print("First English sentence:", test_ds[0]["en"])
print("First Hindi reference:", test_ds[0]["hi"])

Map:   0%|          | 0/2507 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2507 [00:00<?, ? examples/s]

Official test examples available: 2507
First English sentence: A black box in your car?
First Hindi reference: आपकी कार में ब्लैक बॉक्स?


In [ ]:
import time
import pandas as pd
import torch
from sacrebleu.metrics import BLEU, CHRF

gemma_model.eval()
gemma_model.config.use_cache = True

gemma_eval_data = test_ds.select(range(100))
gemma_lora_predictions = []
gemma_references = []

start_time = time.time()

for i, example in enumerate(gemma_eval_data):
    messages = [{
        "role": "user",
        "content": (
            "Translate the following English sentence into natural Hindi. "
            "Return only the Hindi translation.\n\n"
            f"English: {example['en']}"
        )
    }]

    inputs = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(gemma_model.device)

    with torch.inference_mode():
        outputs = gemma_model.generate(
            inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id,
            eos_token_id=gemma_tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs.shape[-1]:]
    prediction = gemma_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    gemma_lora_predictions.append(prediction)
    gemma_references.append(example["hi"])

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/100")

gemma_lora_inference_seconds = time.time() - start_time

gemma_lora_bleu = BLEU().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_chrf = CHRF().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_trainable = sum(
    p.numel() for p in gemma_model.parameters() if p.requires_grad
)
gemma_lora_total = sum(
    p.numel() for p in gemma_model.parameters()
)

gemma_lora_results = {
    "model": "Gemma-2-2B-IT LoRA",
    "test_examples": 100,
    "BLEU": gemma_lora_bleu,
    "chrF": gemma_lora_chrf,
    "training_time_minutes": gemma_lora_training_seconds / 60,
    "inference_time_seconds": gemma_lora_inference_seconds,
    "average_inference_seconds": gemma_lora_inference_seconds / 100,
    "trainable_parameters": gemma_lora_trainable,
    "total_parameters": gemma_lora_total
}

pd.DataFrame({
    "English": [x["en"] for x in gemma_eval_data],
    "Reference_Hindi": gemma_references,
    "Gemma_LoRA_Hindi": gemma_lora_predictions
}).to_csv("gemma_lora_predictions.csv", index=False)

pd.DataFrame([gemma_lora_results]).to_csv(
    "gemma_lora_results.csv", index=False
)

print("\n===== GEMMA LoRA RESULTS =====")
for key, value in gemma_lora_results.items():
    print(f"{key}: {value}")

print("\n===== FIVE SAMPLE TRANSLATIONS =====")
for i in range(5):
    print(f"\nEnglish: {gemma_eval_data[i]['en']}")
    print(f"Reference: {gemma_references[i]}")
    print(f"Prediction: {gemma_lora_predictions[i]}")

AttributeError: 

In [ ]:
import time
import pandas as pd
import torch
from sacrebleu.metrics import BLEU, CHRF

gemma_model.eval()
gemma_model.config.use_cache = True

gemma_eval_data = test_ds.select(range(100))
gemma_lora_predictions = []
gemma_references = []

start_time = time.time()

for i, example in enumerate(gemma_eval_data):
    messages = [{
        "role": "user",
        "content": (
            "Translate the following English sentence into natural Hindi. "
            "Return only the Hindi translation.\n\n"
            f"English: {example['en']}"
        )
    }]

    # Convert the chat template output explicitly into a tensor
    input_ids = gemma_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Some tokenizer versions return a BatchEncoding instead
    if isinstance(input_ids, dict) or hasattr(input_ids, "input_ids"):
        input_ids = input_ids["input_ids"]

    input_ids = input_ids.to(gemma_model.device)

    with torch.inference_mode():
        outputs = gemma_model.generate(
            input_ids=input_ids,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=gemma_tokenizer.pad_token_id,
            eos_token_id=gemma_tokenizer.eos_token_id
        )

    new_tokens = outputs[0, input_ids.shape[-1]:]
    prediction = gemma_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    gemma_lora_predictions.append(prediction)
    gemma_references.append(example["hi"])

    if (i + 1) % 10 == 0:
        print(f"Evaluated {i + 1}/100")

gemma_lora_inference_seconds = time.time() - start_time

gemma_lora_bleu = BLEU().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_chrf = CHRF().corpus_score(
    gemma_lora_predictions, [gemma_references]
).score

gemma_lora_trainable = sum(
    p.numel() for p in gemma_model.parameters() if p.requires_grad
)
gemma_lora_total = sum(
    p.numel() for p in gemma_model.parameters()
)

gemma_lora_results = {
    "model": "Gemma-2-2B-IT LoRA",
    "test_examples": len(gemma_lora_predictions),
    "BLEU": gemma_lora_bleu,
    "chrF": gemma_lora_chrf,
    "training_time_minutes": gemma_lora_training_seconds / 60,
    "inference_time_seconds": gemma_lora_inference_seconds,
    "average_inference_seconds": (
        gemma_lora_inference_seconds / len(gemma_lora_predictions)
    ),
    "trainable_parameters": gemma_lora_trainable,
    "total_parameters": gemma_lora_total
}

pd.DataFrame({
    "English": [x["en"] for x in gemma_eval_data],
    "Reference_Hindi": gemma_references,
    "Gemma_LoRA_Hindi": gemma_lora_predictions
}).to_csv("gemma_lora_predictions.csv", index=False)

pd.DataFrame([gemma_lora_results]).to_csv(
    "gemma_lora_results.csv", index=False
)

print("\n===== GEMMA LoRA RESULTS =====")
for key, value in gemma_lora_results.items():
    print(f"{key}: {value}")

print("\n===== FIVE SAMPLE TRANSLATIONS =====")
for i in range(min(5, len(gemma_lora_predictions))):
    print(f"\nEnglish: {gemma_eval_data[i]['en']}")
    print(f"Reference: {gemma_references[i]}")
    print(f"Prediction: {gemma_lora_predictions[i]}")

Evaluated 10/100
Evaluated 20/100
Evaluated 30/100
Evaluated 40/100
Evaluated 50/100
Evaluated 60/100
Evaluated 70/100
Evaluated 80/100
Evaluated 90/100
Evaluated 100/100

===== GEMMA LoRA RESULTS =====
model: Gemma-2-2B-IT LoRA
test_examples: 100
BLEU: 9.051852417047296
chrF: 35.920805008790445
training_time_minutes: 14.483038214842479
inference_time_seconds: 438.15261459350586
average_inference_seconds: 4.381526145935059
trainable_parameters: 3194880
total_parameters: 2617536768

===== FIVE SAMPLE TRANSLATIONS =====

English: A black box in your car?
Reference: आपकी कार में ब्लैक बॉक्स?
Prediction: कार में एक काले बॉक्स?

English: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स 

In [ ]:
import pandas as pd

results = [
    {
        "Model": "Transformer from scratch",
        "Approach": "Train from scratch",
        "Test examples": 2507,
        "BLEU": 0.15,
        "chrF": 4.72,
        "Avg inference (s)": 462.96 / 2507,
        "Training time (min)": 3.85,
        "Status": "Completed"
    },
    {
        "Model": "mT5-small",
        "Approach": "Fine-tuning attempt",
        "Test examples": None,
        "BLEU": None,
        "chrF": None,
        "Avg inference (s)": None,
        "Training time (min)": None,
        "Status": "Unstable; invalid evaluation"
    },
    {
        "Model": "TinyLlama 1.1B",
        "Approach": "Zero-shot",
        "Test examples": 100,
        "BLEU": 0.63,
        "chrF": 13.25,
        "Avg inference (s)": 3.08,
        "Training time (min)": 0,
        "Status": "Completed"
    },
    {
        "Model": "TinyLlama 1.1B",
        "Approach": "LoRA",
        "Test examples": 100,
        "BLEU": 0.66,
        "chrF": 12.99,
        "Avg inference (s)": 3.24,
        "Training time (min)": 4.20,
        "Status": "Completed"
    },
    {
        "Model": "Gemma 2 2B",
        "Approach": "Zero-shot",
        "Test examples": 100,
        "BLEU": 12.43,
        "chrF": 38.84,
        "Avg inference (s)": 1.97,
        "Training time (min)": 0,
        "Status": "Completed"
    },
    {
        "Model": "Gemma 2 2B",
        "Approach": "LoRA",
        "Test examples": 100,
        "BLEU": gemma_lora_bleu,
        "chrF": gemma_lora_chrf,
        "Avg inference (s)": gemma_lora_inference_seconds / 100,
        "Training time (min)": gemma_lora_training_seconds / 60,
        "Status": "Completed; scores declined"
    },
    {
        "Model": "OPUS-MT en-hi",
        "Approach": "Pretrained translation baseline",
        "Test examples": 2507,
        "BLEU": 9.88,
        "chrF": 32.33,
        "Avg inference (s)": 739.94 / 2507,
        "Training time (min)": None,
        "Status": "Additional baseline"
    }
]

comparison_df = pd.DataFrame(results)
comparison_df.to_csv("model_comparison_results.csv", index=False)

display(comparison_df.round(3))
print("\nSaved: model_comparison_results.csv")

,Model,Approach,Test examples,BLEU,chrF,Avg inference (s),Training time (min),Status
0,Transformer from scratch,Train from scratch,2507.0,0.150,4.720,0.185,3.850,Completed
1,mT5-small,Fine-tuning attempt,NaN,NaN,NaN,NaN,NaN,Unstable; invalid evaluation
2,TinyLlama 1.1B,Zero-shot,100.0,0.630,13.250,3.080,0.000,Completed
3,TinyLlama 1.1B,LoRA,100.0,0.660,12.990,3.240,4.200,Completed
4,Gemma 2 2B,Zero-shot,100.0,12.430,38.840,1.970,0.000,Completed
5,Gemma 2 2B,LoRA,100.0,9.052,35.921,4.382,14.483,Completed; scores declined
6,OPUS-MT en-hi,Pretrained translation baseline,2507.0,9.880,32.330,0.295,NaN,Additional baseline



Saved: model_comparison_results.csv


In [ ]:
import os
import json
import zipfile
import pandas as pd

output_dir = "FGAI_Assignment3_Results"
os.makedirs(output_dir, exist_ok=True)

# Save the comparison table
comparison_df.to_csv(
    os.path.join(output_dir, "model_comparison_results.csv"),
    index=False
)

# Save the Gemma LoRA metrics
with open(os.path.join(output_dir, "gemma_lora_results.json"), "w") as f:
    json.dump(gemma_lora_results, f, indent=2)

# Copy generated result/prediction CSVs if they exist
files_to_collect = [
    "gemma_lora_predictions.csv",
    "gemma_lora_results.csv",
    "gemma_zero_shot_predictions.csv",
    "tinyllama_lora_predictions.csv",
    "opus_mt_predictions.csv",
    "opus_mt_results.csv"
]

for filename in files_to_collect:
    if os.path.exists(filename):
        pd.read_csv(filename).to_csv(
            os.path.join(output_dir, filename),
            index=False
        )
        print("Collected:", filename)
    else:
        print("Not found in current session:", filename)

# Include the Transformer checkpoint if it is still available
if os.path.exists("transformer_corrected_best.pt"):
    import shutil
    shutil.copy2(
        "transformer_corrected_best.pt",
        os.path.join(output_dir, "transformer_corrected_best.pt")
    )
    print("Collected Transformer checkpoint")

zip_path = "FGAI_Assignment3_Results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(output_dir):
        for filename in files:
            full_path = os.path.join(root, filename)
            z.write(full_path, arcname=os.path.relpath(full_path, "."))

print("\nResults folder:", output_dir)
print("ZIP created:", zip_path)
print("Files included:")
for filename in os.listdir(output_dir):
    print(" -", filename)

Collected: gemma_lora_predictions.csv
Collected: gemma_lora_results.csv
Collected: gemma_zero_shot_predictions.csv
Collected: tinyllama_lora_predictions.csv
Collected: opus_mt_predictions.csv
Not found in current session: opus_mt_results.csv
Collected Transformer checkpoint

Results folder: FGAI_Assignment3_Results
ZIP created: FGAI_Assignment3_Results.zip
Files included:
 - gemma_zero_shot_predictions.csv
 - transformer_corrected_best.pt
 - gemma_lora_predictions.csv
 - model_comparison_results.csv
 - gemma_lora_results.csv
 - tinyllama_lora_predictions.csv
 - opus_mt_predictions.csv
 - gemma_lora_results.json


In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError("Add GITHUB_TOKEN to Colab Secrets first.")

GITHUB_USERNAME = input("Enter your GitHub username: ").strip()
REPO_NAME = "FGAI-Assignment-3-English-Hindi-Translation"

print("GitHub credentials loaded securely.")
print("Repository:", REPO_NAME)

TimeoutException: Requesting secret GITHUB_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
from google.colab import userdata
import time

GITHUB_TOKEN = None

for attempt in range(3):
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
        if GITHUB_TOKEN:
            print("GitHub token retrieved successfully.")
            break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {type(e).__name__}")
        if attempt < 2:
            time.sleep(3)

if not GITHUB_TOKEN:
    print("Secrets are still unavailable. Do not run the Git push cell yet.")
else:
    GITHUB_USERNAME = input("Enter your GitHub username: ").strip()
    REPO_NAME = "FGAI-Assignment-3-English-Hindi-Translation"
    print("Ready to configure GitHub push.")

GitHub token retrieved successfully.
Enter your GitHub username: Rishabh-1011
Ready to configure GitHub push.


In [ ]:
import os

repo_dir = "FGAI_Assignment3"

print("Repository folder exists:", os.path.exists(repo_dir))

for root, dirs, files in os.walk(repo_dir):
    for filename in files:
        path = os.path.join(root, filename)
        print(f"{path} ({os.path.getsize(path) / 1024:.1f} KB)")

print("\nCurrent directory files:")
for filename in os.listdir("."):
    if filename.endswith(".ipynb"):
        print(filename)

Repository folder exists: False

Current directory files:


In [ ]:
import os

print("Current directory:", os.getcwd())

# Check common Colab locations
locations = [
    "/content",
    "/content/drive/MyDrive",
    "/content/FGAI_Assignment3_Results",
    "/content/FGAI_Assignment3_Results.zip"
]

for path in locations:
    print(f"\n{path}:")
    if os.path.isfile(path):
        print(f"  FILE FOUND ({os.path.getsize(path) / 1024:.1f} KB)")
    elif os.path.isdir(path):
        items = os.listdir(path)
        print(f"  {len(items)} items")
        for item in items[:20]:
            print("  -", item)
    else:
        print("  Not found")

Current directory: /content

/content:
  2 items
  - .config
  - sample_data

/content/drive/MyDrive:
  Not found

/content/FGAI_Assignment3_Results:
  Not found

/content/FGAI_Assignment3_Results.zip:
  Not found


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for filename in files:
        if filename == 'English_Hindi_Dataset_Preprocessing.ipynb':
            print(os.path.join(root, filename))

/content/drive/MyDrive/Colab Notebooks/English_Hindi_Dataset_Preprocessing.ipynb
